In [ ]:
from sqlite3 import connect
from glob import glob
from collections import Counter
from os import remove
import numpy as np
import torch
from torch.nn import *
class Res(Sequential):
    def forward(s, x):
        return super().forward(x)+x
class At(MultiheadAttention):
    def forward(s, x):
        return super().forward(x, x, x, attn_mask=Transformer.generate_square_subsequent_mask(x.size(-2),dtype=torch.half,device=x.device), is_causal=True)[0]
class Swiglu(Linear):
    def __init__(s, i, o=0):
        super().__init__(i, 2*(o or i//6*16))
        s._ = Linear(o or i//6*16, i)
    def forward(s, x):
        x,_ = super().forward(x).chunk(2,-1)
        return s._(x*functional.silu(_))
class REnCoder(Module):
    def __init__(s, o, *i):
        super().__init__()
        _ = 0
        for j,i in enumerate(i):
            if isinstance(i,(int,float)):
                setattr(s, str(j), (Embedding if isinstance(i,int)else EmbeddingBag)(int(i),o,scale_grad_by_freq=True))
            else:
                _ += 1
        s._ = Linear(_, o)
    def forward(s, x):
        o = 0
        _ = []
        for i in range(x.size(-1)):
            if isinstance(m:=getattr(s,str(i),None), Embedding):
                o += m(x[...,i])
            elif isinstance(m, EmbeddingBag):
                o += (1<<torch.arange(m.num_embeddings,dtype=x.dtype,device=x.device)&x[...,i,None]).bool().to(m.weight.dtype)@m.weight
            else:
                _ += [i]
        return o + s._(x[...,_].to(s._.weight.dtype))
RETRAIN = False
try:
    MAP = dict(map(reversed,enumerate(MAP_:=np.fromfile('model4a.i','i').tolist())))
except FileNotFoundError as _:
    print(_)
    RETRAIN = True
    try:
        remove('model4a.pk')
    except FileNotFoundError:
        pass
    MAP = {}
CARDS = {id:(
    alias or id,
    ot-(ot>0x4)*0x4,
    (setcode&0xfff).bit_length(),
    setcode>>12&0xf,
    (setcode>>16&0xfff).bit_length(),
    setcode>>28&0xf,
    type,
    atk,
    0 if type&0x4000000 else Def,
    level&15,
    race.bit_length(),
    attribute.bit_length(),
    category,
    level>>24,
    Def-(Def>0x10)*0x10 if type&0x4000000 else 0,
    len(name),
    len(desc),
    sum(bool(j in desc or j in name)<<i for i,j in enumerate('·不为之击在地坏外手方时有特表魔'))
)for id,ot,alias,setcode,type,atk,Def,level,race,attribute,category,name,desc in connect('cards.cdb').execute('select datas.*,name,desc from datas join texts on datas.id=texts.id')}
DECK = [torch.tensor([(i,min(n,3))for i,n in Counter(MAP.setdefault(CARDS[int(i)][0],len(MAP))for i in open(i).read().split('!')[0].split()if i.isdigit()and int(i)in(RETRAIN and CARDS or MAP)).items()],dtype=torch.short,device='cuda')for i in glob('deck/**/**.ydk',recursive=True,include_hidden=True)]
DECK = [i for i in DECK if i[...,-1:].sum()>=40]
CARD = torch.tensor([CARDS[i][1:]for i in MAP],dtype=torch.uint32,device='cuda').view(torch.int)
np.array(list(MAP),'i').tofile('model4a.i')

In [ ]:
try:
    m,o,s = torch.load('model4a.pk','cuda',weights_only=False)
except FileNotFoundError as _:
    print(_)
    m = 256
    m = torch.compile(Sequential(
        REnCoder(m,len(MAP),int,3.,475,4.,475,4.,27.,int,int,int,27,8,32.,int,8.,int,int,16.),
        *(Sequential(
            Res(
                LayerNorm(m),
                At(m, 32, batch_first=True),
            ),
            Res(
                LayerNorm(m),
                Swiglu(m),
            ),
        )for _ in range(5)),
        Linear(m, len(MAP))
    ).cuda())
    o = torch.optim.Adam(m.parameters(), .0008)
    s = torch.amp.GradScaler()    

In [ ]:
while True:
    L = 0
    for _ in range(10000):
        o.zero_grad()
        a = [DECK[i][torch.randperm(len(DECK[i]),dtype=torch.int,device='cuda')]for i in torch.randint(len(DECK),[8],dtype=torch.short)]
        x = torch.stack([torch.constant_pad_nd(torch.cat([i,CARD[i[:,0].int()]],1),(0,0,0,max(map(len,a))-len(i)))for i in a])
        with torch.amp.autocast('cuda', torch.float16):
            y = m(x[:,:-1]).log_softmax(2)
            t = functional.one_hot((x[:,1:,1].bool()+x[:,1:,0]).flip(1).long(),1+len(MAP))[:,:,1:].half().cumsum_(1).flip(1).sign_().log_()
            t[t.all(2)] = 0
            l = -(y+t).logsumexp(2).mean()
        s.scale(l).backward()
        s.step(o)
        s.update()
        L += l.data.float()
    print(L)
    torch.save((m,o,s),'model4a.pk')